# 泛化性实验 — 算法展开的核心挑战

本 notebook 系统性地研究展开网络的泛化性，回答以下问题：

1. **同一 A 矩阵，不同 x**：标准设定下性能如何？
2. **不同 A 矩阵**：能否泛化到未见过的测量矩阵？
3. **不同问题维度**：n 变化时性能如何？
4. **不同稀疏度**：s 变化时性能如何？
5. **不同条件数**：κ(A) 变化时性能如何？
6. **混合训练**：在多个 A 上训练能否提升泛化性？

In [ ]:
import sys, os
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath('.'))))

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from tqdm import tqdm
import itertools

from common.utils import set_seed, get_device, to_numpy, count_parameters
from common.metrics import relative_error
from common.visualization import setup_figure
from common.training import Trainer
from lasso.lista import LISTA, LISTACP, LISTACPFISTA, create_lista
from lasso.classical import ista, fista
from lasso.problem import generate_lasso_data

set_seed(42)
device = get_device()
print(f'Device: {device}')

# 基础参数
m_default, n_default = 50, 200
sparsity_default = 10
noise_default = 0.01

## 0. 工具函数

In [ ]:
def generate_shared_A(m, n, seed=42):
    """生成共享的 A 矩阵。"""
    rng = np.random.RandomState(seed)
    A = rng.randn(m, n)
    A /= np.linalg.norm(A, axis=0, keepdims=True)
    return A

def generate_dataset(A, num_samples, sparsity=10, noise_std=0.01, seed=0):
    """生成数据集。"""
    rng = np.random.RandomState(seed)
    m, n = A.shape
    B, X = [], []
    for _ in range(num_samples):
        x = np.zeros(n)
        support = rng.choice(n, sparsity, replace=False)
        x[support] = rng.randn(sparsity)
        b = A @ x + noise_std * rng.randn(m)
        B.append(b)
        X.append(x)
    return np.array(B), np.array(X)

def train_on_A(A, T=10, variant='cp', num_train=500, num_epochs=50, seed=42):
    """在单个 A 矩阵上训练模型。"""
    set_seed(seed)
    m, n = A.shape
    
    # 生成训练数据
    B_train, X_train = generate_dataset(A, num_train, seed=seed)
    B_val, X_val = generate_dataset(A, 100, seed=seed+1)
    
    train_loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(torch.FloatTensor(B_train), torch.FloatTensor(X_train)),
        batch_size=64, shuffle=True)
    val_loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(torch.FloatTensor(B_val), torch.FloatTensor(X_val)),
        batch_size=64)
    
    # 创建模型
    A_tensor = torch.FloatTensor(A)
    model = create_lista(A_tensor, variant=variant, T=T)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)
    
    trainer = Trainer(model=model, optimizer=optimizer, criterion=nn.MSELoss(),
                      scheduler=scheduler, grad_clip=1.0, patience=15, device=device, verbose=False)
    
    def forward_fn(model, batch):
        b, x = batch
        return model(b), x
    
    trainer.fit(train_loader, val_loader, num_epochs, forward_fn)
    return model

def evaluate_on_A(model, A, num_test=100, sparsity=10, noise_std=0.01, seed=999):
    """在指定 A 上评估模型。"""
    model.eval()
    B_test, X_test = generate_dataset(A, num_test, sparsity, noise_std, seed)
    errors = []
    with torch.no_grad():
        for i in range(num_test):
            b_tensor = torch.FloatTensor(B_test[i:i+1]).to(device)
            x_pred = model(b_tensor)
            errors.append(relative_error(X_test[i], to_numpy(x_pred.squeeze())))
    return np.array(errors)

print('工具函数定义完成')

## 1. 基准实验：同一 A 矩阵，不同 x

标准设定：训练和测试使用同一个 A 矩阵，但不同的稀疏信号 x。
这是最理想的情况，也是大多数论文报告的结果。

In [ ]:
# 基准实验
A_base = generate_shared_A(m_default, n_default)

print('训练 LISTA-CP...')
model_base = train_on_A(A_base, T=10, variant='cp', seed=42)

# 同一 A，不同 x
errors_same_A = evaluate_on_A(model_base, A_base, num_test=200)

# ISTA baseline
ista_errors = []
B_test, X_test = generate_dataset(A_base, 200, seed=999)
for i in range(200):
    lam = 0.1 * np.max(np.abs(A_base.T @ B_test[i]))
    x_ista, _ = ista(A_base, B_test[i], lam, max_iter=100)
    ista_errors.append(relative_error(X_test[i], x_ista))

print(f'\n基准实验结果 (同一 A，不同 x):')
print(f'  LISTA-CP: {np.mean(errors_same_A):.6f} ± {np.std(errors_same_A):.6f}')
print(f'  ISTA:     {np.mean(ista_errors):.6f} ± {np.std(ista_errors):.6f}')

## 2. 核心实验：泛化到不同的 A 矩阵

这是算法展开最关键的泛化性问题：
训练时用 A_train，测试时用完全不同的 A_test。

**问题**：展开网络学习的是 A_train 的特定结构，还是通用的稀疏恢复能力？

In [ ]:
# 实验：泛化到不同的 A 矩阵
num_A_test = 20  # 测试 20 个不同的 A
errors_diff_A = []
errors_diff_A_ista = []

print('测试泛化到不同 A 矩阵...')
for i in tqdm(range(num_A_test)):
    # 生成不同的 A
    A_new = generate_shared_A(m_default, n_default, seed=1000+i)
    
    # LISTA-CP (在 A_base 上训练)
    errors = evaluate_on_A(model_base, A_new, num_test=50, seed=2000+i)
    errors_diff_A.append(np.mean(errors))
    
    # ISTA baseline
    B_test, X_test = generate_dataset(A_new, 50, seed=2000+i)
    ista_errs = []
    for j in range(50):
        lam = 0.1 * np.max(np.abs(A_new.T @ B_test[j]))
        x_ista, _ = ista(A_new, B_test[j], lam, max_iter=100)
        ista_errs.append(relative_error(X_test[j], x_ista))
    errors_diff_A_ista.append(np.mean(ista_errs))

print(f'\n泛化到不同 A 的结果:')
print(f'  LISTA-CP (在 A_base 上训练): {np.mean(errors_diff_A):.6f} ± {np.std(errors_diff_A):.6f}')
print(f'  ISTA (无需训练):             {np.mean(errors_diff_A_ista):.6f} ± {np.std(errors_diff_A_ista):.6f}')
print(f'\n  相比基准 (同一 A) 的退化: {np.mean(errors_diff_A) / np.mean(errors_same_A):.1f}x')

In [ ]:
# 可视化
fig, ax = setup_figure(figsize=(10, 5))

data = [errors_same_A, errors_diff_A, errors_diff_A_ista]
labels = ['Same A\n(In-Distribution)', 'Different A\n(Out-of-Distribution)', 'ISTA\n(No Training)']
colors = ['#2ca02c', '#d62728', '#1f77b4']

bp = ax.boxplot(data, labels=labels, patch_artist=True)
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_ylabel('Relative Error', fontsize=12)
ax.set_title('Generalization to Different Measurement Matrices A', fontsize=14)
ax.set_yscale('log')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('generalization_A.png', dpi=300, bbox_inches='tight')
plt.show()

## 3. 不同问题维度 n

训练在 n=200 上，测试在 n=100, 150, 200, 250, 300 上。

**注意**：当 n 变化时，网络结构也必须变化 (W₁, W₂ 的维度变了)。
所以我们测试的是：在 n=200 上训练的网络，能否直接用于不同 n？

In [ ]:
# 实验：不同问题维度
n_values = [100, 150, 200, 250, 300]
errors_by_n = {}
errors_by_n_ista = {}

print('测试不同问题维度...')
for n in tqdm(n_values):
    A_n = generate_shared_A(m_default, n, seed=42)
    
    if n == n_default:
        # 直接使用基准模型
        model_n = model_base
    else:
        # n 不同，网络维度不匹配，无法使用
        # 这本身就是泛化性的限制！
        model_n = None
    
    # ISTA baseline (总能工作)
    B_test, X_test = generate_dataset(A_n, 100, seed=999)
    ista_errs = []
    for j in range(100):
        lam = 0.1 * np.max(np.abs(A_n.T @ B_test[j]))
        x_ista, _ = ista(A_n, B_test[j], lam, max_iter=100)
        ista_errs.append(relative_error(X_test[j], x_ista))
    errors_by_n_ista[n] = np.mean(ista_errs)
    
    if model_n is not None:
        errors = evaluate_on_A(model_n, A_n, num_test=100, seed=999)
        errors_by_n[n] = np.mean(errors)
    else:
        errors_by_n[n] = None  # 无法评估

print('\n不同问题维度 n 的结果:')
print(f'{"n":>6} {"LISTA-CP":>12} {"ISTA":>12} {"Notes":>20}')
print('-' * 55)
for n in n_values:
    lista_str = f'{errors_by_n[n]:.6f}' if errors_by_n[n] is not None else 'N/A (dim mismatch)'
    print(f'{n:>6} {lista_str:>12} {errors_by_n_ista[n]:>12.6f} {"" if n == n_default else "← OOD"}')

## 4. 不同稀疏度 s

训练在 s=10 上，测试在 s=5, 10, 15, 20, 30 上。

稀疏度变化不影响网络结构，但影响信号的统计特性。

In [ ]:
# 实验：不同稀疏度
sparsity_values = [5, 10, 15, 20, 30]
errors_by_s = {}
errors_by_s_ista = {}

print('测试不同稀疏度...')
for s in tqdm(sparsity_values):
    # LISTA-CP (在 s=10 上训练)
    errors = evaluate_on_A(model_base, A_base, num_test=100, sparsity=s, seed=999)
    errors_by_s[s] = np.mean(errors)
    
    # ISTA baseline
    B_test, X_test = generate_dataset(A_base, 100, sparsity=s, seed=999)
    ista_errs = []
    for j in range(100):
        lam = 0.1 * np.max(np.abs(A_base.T @ B_test[j]))
        x_ista, _ = ista(A_base, B_test[j], lam, max_iter=100)
        ista_errs.append(relative_error(X_test[j], x_ista))
    errors_by_s_ista[s] = np.mean(ista_errs)

print('\n不同稀疏度 s 的结果 (训练时 s=10):')
print(f'{"s":>6} {"LISTA-CP":>12} {"ISTA":>12} {"Notes":>20}')
print('-' * 55)
for s in sparsity_values:
    note = '← training' if s == 10 else ('← easier' if s < 10 else '← harder')
    print(f'{s:>6} {errors_by_s[s]:>12.6f} {errors_by_s_ista[s]:>12.6f} {note}')

In [ ]:
# 可视化稀疏度实验
fig, ax = setup_figure(figsize=(8, 5))

ax.plot(sparsity_values, [errors_by_s[s] for s in sparsity_values], 'o-', 
        label='LISTA-CP (trained at s=10)', color='#d62728', markersize=8, linewidth=2)
ax.plot(sparsity_values, [errors_by_s_ista[s] for s in sparsity_values], 's-', 
        label='ISTK (no training)', color='#1f77b4', markersize=8, linewidth=2)

ax.axvline(x=10, color='gray', linestyle='--', alpha=0.5, label='Training sparsity')
ax.set_xlabel('Sparsity Level s', fontsize=12)
ax.set_ylabel('Relative Error', fontsize=12)
ax.set_title('Generalization Across Sparsity Levels', fontsize=14)
ax.set_yscale('log')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('generalization_sparsity.png', dpi=300, bbox_inches='tight')
plt.show()

## 5. 不同条件数 κ(A)

条件数 κ(A) = σ_max / σ_min 衡量矩阵的病态程度。
训练在 κ ≈ 1 上，测试在不同 κ 上。

In [ ]:
from common.numerical import generate_spd_matrix

def generate_A_with_condition(m, n, kappa, seed=42):
    """生成指定条件数的 A 矩阵。"""
    rng = np.random.RandomState(seed)
    # 通过 SVD 控制条件数
    U, _ = np.linalg.qr(rng.randn(m, m))
    V, _ = np.linalg.qr(rng.randn(n, n))
    sigma = np.linspace(1.0, 1.0/kappa, min(m, n))
    S = np.zeros((m, n))
    np.fill_diagonal(S, sigma)
    A = U @ S @ V.T
    A /= np.linalg.norm(A, axis=0, keepdims=True)
    return A

# 实验：不同条件数
kappa_values = [1, 2, 5, 10, 20, 50]
errors_by_kappa = {}
errors_by_kappa_ista = {}

print('测试不同条件数...')
for kappa in tqdm(kappa_values):
    A_kappa = generate_A_with_condition(m_default, n_default, kappa, seed=42)
    
    # 需要重新训练 (因为 A 变了)
    model_kappa = train_on_A(A_kappa, T=10, variant='cp', seed=42)
    
    # LISTA-CP
    errors = evaluate_on_A(model_kappa, A_kappa, num_test=100, seed=999)
    errors_by_kappa[kappa] = np.mean(errors)
    
    # ISTA baseline
    B_test, X_test = generate_dataset(A_kappa, 100, seed=999)
    ista_errs = []
    for j in range(100):
        lam = 0.1 * np.max(np.abs(A_kappa.T @ B_test[j]))
        x_ista, _ = ista(A_kappa, B_test[j], lam, max_iter=100)
        ista_errs.append(relative_error(X_test[j], x_ista))
    errors_by_kappa_ista[kappa] = np.mean(ista_errs)

print('\n不同条件数 κ 的结果:')
print(f'{"κ":>6} {"LISTA-CP":>12} {"ISTA":>12}')
print('-' * 35)
for kappa in kappa_values:
    print(f'{kappa:>6} {errors_by_kappa[kappa]:>12.6f} {errors_by_kappa_ista[kappa]:>12.6f}')

In [ ]:
# 可视化条件数实验
fig, ax = setup_figure(figsize=(8, 5))

ax.plot(kappa_values, [errors_by_kappa[k] for k in kappa_values], 'o-', 
        label='LISTA-CP', color='#d62728', markersize=8, linewidth=2)
ax.plot(kappa_values, [errors_by_kappa_ista[k] for k in kappa_values], 's-', 
        label='ISTA', color='#1f77b4', markersize=8, linewidth=2)

ax.set_xlabel('Condition Number κ(A)', fontsize=12)
ax.set_ylabel('Relative Error', fontsize=12)
ax.set_title('Sensitivity to Matrix Condition Number', fontsize=14)
ax.set_yscale('log')
ax.set_xscale('log')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('generalization_kappa.png', dpi=300, bbox_inches='tight')
plt.show()

## 6. 混合训练：在多个 A 上训练

**关键问题**：能否通过在多个不同的 A 上训练来提升泛化性？

方法：训练时每个 batch 使用不同的 A 矩阵。

In [ ]:
def train_multi_A(m, n, T=10, variant='cp', num_A=50, num_per_A=20, num_epochs=100, seed=42):
    """在多个 A 矩阵上训练。"""
    set_seed(seed)
    
    # 生成多个 A 和对应的数据
    all_B, all_X = [], []
    A_list = []
    for i in range(num_A):
        A = generate_shared_A(m, n, seed=seed+i)
        A_list.append(A)
        B, X = generate_dataset(A, num_per_A, seed=seed+1000+i)
        all_B.append(B)
        all_X.append(X)
    
    all_B = np.concatenate(all_B)
    all_X = np.concatenate(all_X)
    
    # 使用平均 A 初始化 (这是关键设计选择)
    A_mean = np.mean(A_list, axis=0)
    A_tensor = torch.FloatTensor(A_mean)
    
    train_loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(torch.FloatTensor(all_B), torch.FloatTensor(all_X)),
        batch_size=128, shuffle=True)
    val_loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(torch.FloatTensor(all_B[:200]), torch.FloatTensor(all_X[:200])),
        batch_size=128)
    
    model = create_lista(A_tensor, variant=variant, T=T)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=10, factor=0.5)
    
    trainer = Trainer(model=model, optimizer=optimizer, criterion=nn.MSELoss(),
                      scheduler=scheduler, grad_clip=1.0, patience=20, device=device, verbose=False)
    
    def forward_fn(model, batch):
        b, x = batch
        return model(b), x
    
    trainer.fit(train_loader, val_loader, num_epochs, forward_fn)
    return model, A_list

# 训练混合模型
print('在多个 A 上训练...')
model_multi, A_list_train = train_multi_A(m_default, n_default, num_A=50, num_per_A=20, seed=42)

# 评估：在训练集中的 A 上
errors_multi_in = []
for i in range(10):  # 10 个训练集中的 A
    errors = evaluate_on_A(model_multi, A_list_train[i], num_test=50, seed=999)
    errors_multi_in.append(np.mean(errors))

# 评估：在训练集外的 A 上
errors_multi_out = []
for i in range(20):  # 20 个新的 A
    A_new = generate_shared_A(m_default, n_default, seed=2000+i)
    errors = evaluate_on_A(model_multi, A_new, num_test=50, seed=3000+i)
    errors_multi_out.append(np.mean(errors))

print(f'\n混合训练结果:')
print(f'  训练集内 A: {np.mean(errors_multi_in):.6f} ± {np.std(errors_multi_in):.6f}')
print(f'  训练集外 A: {np.mean(errors_multi_out):.6f} ± {np.std(errors_multi_out):.6f}')
print(f'\n对比单 A 训练:')
print(f'  同一 A (ID):   {np.mean(errors_same_A):.6f}')
print(f'  不同 A (OOD):  {np.mean(errors_diff_A):.6f}')

In [ ]:
# 综合对比图
fig, ax = setup_figure(figsize=(12, 6))

data = [
    errors_same_A,
    errors_diff_A,
    errors_multi_in,
    errors_multi_out,
    errors_diff_A_ista,
]
labels = [
    'Single A\n(ID)',
    'Single A\n(OOD)',
    'Multi A\n(In-Train)',
    'Multi A\n(Out-Train)',
    'ISTA\n(No Train)',
]
colors = ['#2ca02c', '#d62728', '#ff7f0e', '#9467bd', '#1f77b4']

bp = ax.boxplot(data, labels=labels, patch_artist=True)
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_ylabel('Relative Error', fontsize=12)
ax.set_title('Comprehensive Generalization Analysis', fontsize=14)
ax.set_yscale('log')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('generalization_comprehensive.png', dpi=300, bbox_inches='tight')
plt.show()

## 7. 结果汇总表

In [ ]:
# 汇总表
print('\n' + '='*70)
print('Table: Generalization Analysis Summary')
print('='*70)
print(f'{"Experiment":<30} {"LISTA-CP":>12} {"ISTA":>12} {"Verdict":>15}')
print('-'*70)

rows = [
    ('Same A (ID)', np.mean(errors_same_A), np.mean(ista_errors), 'Baseline'),
    ('Different A (OOD)', np.mean(errors_diff_A), np.mean(errors_diff_A_ista), 'FAIL'),
    ('Multi-A In-Train', np.mean(errors_multi_in), np.mean(ista_errors), 'Partial'),
    ('Multi-A Out-Train', np.mean(errors_multi_out), np.mean(errors_diff_A_ista), 'Partial'),
    ('s=5 (easier)', errors_by_s[5], errors_by_s_ista[5], 'OK'),
    ('s=20 (harder)', errors_by_s[20], errors_by_s_ista[20], 'Degraded'),
    ('s=30 (much harder)', errors_by_s[30], errors_by_s_ista[30], 'FAIL'),
]

for name, lista_err, ista_err, verdict in rows:
    print(f'{name:<30} {lista_err:>12.6f} {ista_err:>12.6f} {verdict:>15}')

print('\n' + '='*70)
print('Key Insight: LISTA-CP excels in-distribution but struggles with OOD data.')
print('='*70)

## 8. 讨论：为什么泛化性差？

### 8.1 根本原因

LISTA-CP 的更新公式：
$$x_{k+1} = \sigma(\eta Bb + (I - \eta BA)x_k; \theta_k)$$

其中 $B$ 是可学习参数，初始化为 $A^T$。

**关键观察**：$B$ 会收敛到 $A^T$ 的某种变形，这种变形是 **特定于训练矩阵 $A$ 的**。

当测试时用不同的 $A_{test}$：
- 网络的 $B$ 仍然对应训练时的 $A_{train}$
- $B \cdot A_{test} \neq B \cdot A_{train}$
- 更新公式不再有意义

### 8.2 类比理解

```
训练: 学会用 A_train 的逆来解题
测试: 给你 A_test，但你还在用 A_train 的逆 → 错误答案
```

这就像：
- 训练时学会用钥匙 A 开锁 A
- 测试时给你锁 B，但你还在用钥匙 A → 开不了

### 8.3 对比经典算法

ISTA 每步都用 **当前的** A 计算梯度：
$$x_{k+1} = \text{SoftThreshold}(x_k - \eta A^T(Ax_k - b), \eta\lambda)$$

所以 ISTA 天然泛化到任何 A。

### 8.4 可能的改进方向

1. **元学习 (Meta-Learning)**：学习一个能快速适应新 A 的初始化
2. **条件网络**：将 A 作为额外输入，而非固定在权重中
3. **数据增强**：训练时随机变换 A
4. **混合方法**：用展开网络做初始化，再用经典算法微调

## 9. 结论

### 泛化性实验的关键发现

1. **同一 A (ID)**：LISTA-CP 优于 ISTA，这是展开网络的优势场景

2. **不同 A (OOD)**：LISTA-CP 性能严重退化，甚至不如不训练的 ISTA

3. **稀疏度变化**：
   - 更稀疏 (s < 训练值)：性能 OK
   - 更稠密 (s > 训练值)：性能下降

4. **混合训练**：在多个 A 上训练可以部分缓解 OOD 问题，但不如单 A 训练在 ID 上的表现

5. **根本限制**：展开网络学习的是 **特定问题结构**，而非 **通用优化算法**

### 实际应用建议

| 场景 | 推荐方法 |
|------|----------|
| 同一设备反复采集 (如 MRI) | 展开网络 |
| 问题结构多变 | 经典算法 |
| 需要理论保证 | 经典算法 |
| 极低延迟要求 | 展开网络 |